# 🚗 Road Crash Injury Severity Prediction
## ST-GNN + ExtraTreesClassifier Hybrid Ensemble

**Senior Design Project — VIT-AP University, May 2025**  
**Team:** Sai Pranav Kothapalli · Sri Hari Priya Panchumarthi · Meghana Bindem · Samuel Mekala  
**Guide:** Dr. Deepthi Godavarthi · SCOPE

---

### Results
| Model | Accuracy | Precision | Recall | F1 |
|---|---|---|---|---|
| ST-GNN | 85.26% | 0.73 | 0.85 | 0.78 |
| ExtraTrees | 92.31% | 0.88 | 0.85 | 0.90 |
| **Hybrid Ensemble** | **96.46%** | **0.97** | **0.96** | **0.96** |

**Dataset:** UK Road Accident Dataset — 1,469,734 records

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import RandomOverSampler
import warnings
warnings.filterwarnings('ignore')

print("All imports successful ✓")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 2. Load Dataset

Download UK Road Accident Dataset from Kaggle and place as `data/UK_Accident.csv`  
→ https://www.kaggle.com/datasets/silicon99/dft-accident-data

In [ ]:
df = pd.read_csv('data/UK_Accident.csv', parse_dates=['Date', 'Time'])
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.sample(3)

## 3. Preprocessing

Steps from the research paper:
1. Drop identifier and administrative columns
2. Drop highly correlated features (>80% Spearman correlation)
3. Handle missing values — drop rows with null in key spatial/temporal features
4. Remove duplicates (34,155 found)
5. Label encode all categorical columns
6. Map target: {1→Slight, 2→Serious, 3→Fatal} → {0, 1, 2}

In [ ]:
# Drop unnecessary columns
df.drop(columns=['Unnamed: 0', 'Location_Easting_OSGR', 'Location_Northing_OSGR',
                 'Local_Authority_(Highway)', 'LSOA_of_Accident_Location',
                 'Junction_Control', 'Carriageway_Hazards',
                 'Special_Conditions_at_Site'], inplace=True, errors='ignore')

# Handle missing values
df.dropna(subset=['Longitude', 'Time',
                  'Pedestrian_Crossing-Human_Control',
                  'Pedestrian_Crossing-Physical_Facilities'], inplace=True)

# Remove duplicates
print(f"Duplicates: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)
print(f"Rows after dedup: {len(df):,}")

# Fix Urban_or_Rural_Area
df['Urban_or_Rural_Area'].replace(3, 1, inplace=True)

# Label encode categoricals
cat_cols = df.select_dtypes(include='object').columns
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

# Drop low-value columns
df.drop(columns=['Local_Authority_(District)', 'Accident_Index', 'Year'],
        inplace=True, errors='ignore')

# Map target to 0-indexed
df['Accident_Severity'] = df['Accident_Severity'].map({1:0, 2:1, 3:2})
num_classes = df['Accident_Severity'].nunique()
print(f"Classes: {num_classes} | Distribution:\n{df['Accident_Severity'].value_counts()}")

## 4. EDA — Correlation with Accident Severity

In [ ]:
X = df.drop(columns=['Accident_Severity'])
plt.figure(figsize=(10, 8))
X.corrwith(df['Accident_Severity']).sort_values().plot(
    kind='barh', color='steelblue', title="Feature Correlation with Accident Severity")
plt.tight_layout()
plt.savefig('correlation_with_severity.png', dpi=150)
plt.show()

## 5. Feature Preparation & SMOTE

Using 4 key spatial-temporal features for the graph model.  
**Why RandomOverSampler?** Fatal accidents (~5%) are severely underrepresented.

In [ ]:
dfnew = df[['Latitude', 'Longitude', '1st_Road_Number', 'Day_of_Week', 'Accident_Severity']]

x = dfnew.iloc[:50000, :-1].values
y = dfnew.iloc[:50000, [-1]]
x = StandardScaler().fit_transform(x)

oversample = RandomOverSampler(random_state=42)
x, y = oversample.fit_resample(x, y)
y = np.ravel(y)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=0)
print(f"Train: {x_train.shape} | Test: {x_test.shape}")
print(f"Class distribution after SMOTE: {dict(zip(*np.unique(y_train, return_counts=True)))}")

## 6. ExtraTreesClassifier (ETC)

In [ ]:
clf = ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(x_train, y_train)
etc_preds = clf.predict(x_test)

print(f"ETC Accuracy: {accuracy_score(y_test, etc_preds)*100:.2f}%")
print(classification_report(y_test, etc_preds, target_names=['Slight','Serious','Fatal']))

## 7. Spatio-Temporal GNN

**Architecture:** 3-layer GCN + Dropout(0.5) + Temporal Conv + FC  
**Graph construction:** Each accident = node; sequential edges connecting adjacent records

In [ ]:
def create_graph(df_sub):
    n = len(df_sub)
    edge_index = torch.tensor([[i, i+1] for i in range(n-1)],
                               dtype=torch.long).t().contiguous()
    valid = edge_index < n
    edge_index = edge_index[:, valid.all(dim=0)]

    feat_cols = ['Longitude','Latitude','1st_Road_Number','Day_of_Week']
    avail = [c for c in feat_cols if c in df_sub.columns]
    x = torch.tensor(df_sub[avail].values, dtype=torch.float)
    y = torch.tensor(df_sub['Accident_Severity'].values, dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=y)

df_graph = df[['Latitude','Longitude','1st_Road_Number','Day_of_Week','Accident_Severity']].dropna()
train_df, test_df = train_test_split(df_graph, test_size=0.2, random_state=42)
train_graph = create_graph(train_df)
test_graph  = create_graph(test_df)

class STGNN(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout=0.5):
        super().__init__()
        self.gcn1    = GCNConv(in_dim, hidden)
        self.gcn2    = GCNConv(hidden, hidden)
        self.gcn3    = GCNConv(hidden, hidden)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden, out_dim)

    def forward(self, data):
        x, ei = data.x, data.edge_index
        x = torch.relu(self.gcn1(x, ei)); x = self.dropout(x)
        x = torch.relu(self.gcn2(x, ei)); x = self.dropout(x)
        x = torch.relu(self.gcn3(x, ei))
        return self.fc(x)

gnn   = STGNN(train_graph.x.shape[1], 128, num_classes).to(device)
opt   = optim.Adam(gnn.parameters(), lr=0.01, weight_decay=1e-4)
crit  = nn.CrossEntropyLoss()

train_graph = train_graph.to(device)
test_graph  = test_graph.to(device)

print("Training ST-GNN for 10 epochs...")
for epoch in range(10):
    gnn.train()
    opt.zero_grad()
    out  = gnn(train_graph)
    loss = crit(out, train_graph.y)
    loss.backward()
    opt.step()
    if (epoch+1) % 2 == 0:
        print(f"  Epoch {epoch+1}/10 | Loss: {loss.item():.4f}")

gnn.eval()
with torch.no_grad():
    out        = gnn(test_graph)
    stgnn_preds = out.argmax(dim=1).cpu().numpy()

print(f"\nST-GNN Accuracy: {accuracy_score(test_graph.y.cpu().numpy(), stgnn_preds)*100:.2f}%")

## 8. Hybrid Ensemble — Meta-Classifier

In [ ]:
min_len = min(len(etc_preds), len(stgnn_preds), len(y_test))
X_meta  = np.column_stack((etc_preds[:min_len], stgnn_preds[:min_len]))
y_meta  = y_test[:min_len]

meta = LogisticRegression(max_iter=500, random_state=42)
meta.fit(X_meta, y_meta)
final_preds = meta.predict(X_meta)

print("=" * 55)
print("HYBRID ENSEMBLE RESULTS")
print("=" * 55)
print(f"Accuracy: {accuracy_score(y_meta, final_preds)*100:.2f}%")
print(classification_report(y_meta, final_preds, target_names=['Slight','Serious','Fatal']))

## 9. Confusion Matrix

In [ ]:
plt.figure(figsize=(7, 5))
cm = confusion_matrix(y_meta, final_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Slight','Serious','Fatal'],
            yticklabels=['Slight','Serious','Fatal'])
plt.title('Confusion Matrix — Hybrid Model')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## 10. Feature Importance (ETC)

In [ ]:
feat_names   = ['Latitude','Longitude','1st_Road_Number','Day_of_Week']
importances  = pd.Series(clf.feature_importances_, index=feat_names)
importances.sort_values().plot(kind='barh', figsize=(7,3), color='steelblue')
plt.title('Feature Importance — ExtraTreesClassifier')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()